In [1]:
#imports
import sklearn
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from collections import Counter

In [ ]:
file_path = "paste your file path here"

In [ ]:
feature_length = 4096  # all features have this length

feature_description = {
    'tmmx': tf.io.FixedLenFeature([feature_length], tf.float32),
    'FireMask': tf.io.FixedLenFeature([feature_length], tf.float32),
    'PrevFireMask': tf.io.FixedLenFeature([feature_length], tf.float32),
    'th': tf.io.FixedLenFeature([feature_length], tf.float32),
    'erc': tf.io.FixedLenFeature([feature_length], tf.float32),
    'vs': tf.io.FixedLenFeature([feature_length], tf.float32),
    'elevation': tf.io.FixedLenFeature([feature_length], tf.float32),
    'tmmn': tf.io.FixedLenFeature([feature_length], tf.float32),
    'NDVI': tf.io.FixedLenFeature([feature_length], tf.float32),
    'sph': tf.io.FixedLenFeature([feature_length], tf.float32),
    'pdsi': tf.io.FixedLenFeature([feature_length], tf.float32),
    'pr': tf.io.FixedLenFeature([feature_length], tf.float32),
    'population': tf.io.FixedLenFeature([feature_length], tf.float32),
}

def _parse_function(proto):
    return tf.io.parse_single_example(proto, feature_description)

dataset = tf.data.TFRecordDataset(file_path)
parsed_dataset = dataset.map(_parse_function)

# Define the list of feature names and the label name
features_to_extract = ['tmmx', 'PrevFireMask', 'th', 'erc', 'vs', 'elevation',
                       'tmmn', 'NDVI', 'sph', 'pdsi', 'pr', 'population']
label_to_extract = 'FireMask'

In [6]:
all_features_data = [[],[],[],[],[],[],[],[],[],[],[],[]]
all_labels_data = []

for parsed_record in parsed_dataset:
  for i in range(len(features_to_extract)):
    all_features_data[i].append(parsed_record[features_to_extract[i]].numpy())

    # Extract the label (FireMask) for the current recor
  all_labels_data.append(parsed_record[label_to_extract].numpy())

# Convert the lists of data into numpy arrays
all_features_data = np.array(all_features_data)

X_transposed = all_features_data.transpose(1, 2, 0)
X_final = X_transposed.reshape(-1, 12)

# Convert the lists of data into numpy arrays
X = np.array(X_final)
y = np.array(all_labels_data).flatten()

#Filtering out -1 pixels
mask = y != -1
X = X[mask]
y = y[mask]

# Sanity check
assert set(np.unique(y)).issubset({0, 1}), "Found unexpected labels!"


print(f"Shape of features (X): {X.shape}")
print(f"Shape of labels (y): {y.shape}")

I0000 00:00:1778437859.634995 35350907 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


Shape of features (X): (4007752, 12)
Shape of labels (y): (4007752,)


In [7]:
# Full dataset
#Splits the data into training and testing sets
# Assuming X contains features and y contains target variable (labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (3206201, 12)
X_test shape: (801551, 12)
y_train shape: (3206201,)
y_test shape: (801551,)


In [ ]:
#actual model
lr_model = LogisticRegression(random_state=42)
#lr_model = MultiOutputClassifier(lr_estimator)
lr_model.fit(X_train, y_train)
# Make predictions on the test set
predictions = lr_model.predict(X_test)

print('success')

Visualization & analysis

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

overall_accuracy = accuracy_score(y_test, predictions)

overall_precision = precision_score(y_test, predictions, average = 'macro')
overall_recall = recall_score(y_test, predictions, average = 'macro')
overall_f1 = f1_score(y_test, predictions, average = 'macro')

print(f"Accuracy: {overall_accuracy}")

print(f"Overall Precision: {overall_precision}")
print(f"Overall Recall: {overall_recall}")
print(f"Overall F1 Score: {overall_f1}")

print("Classification Report:\n", classification_report(y_test, predictions))

In [ ]:
#Visualization of predictions for a single record
dataset = tf.data.TFRecordDataset(file_path)
num_records_to_skip = 0
second_parsed_dataset = dataset.map(_parse_function).skip(num_records_to_skip).take(1)

features_data = [[],[],[],[],[],[],[],[],[],[],[],[]]
labels_data = []

for parsed_record in second_parsed_dataset:
  for i in range(len(features_to_extract)):
    features_data[i].append(parsed_record[features_to_extract[i]].numpy())

    # Extract the label (FireMask) for the current recor
  labels_data.append(parsed_record[label_to_extract].numpy())

  fire_mask = parsed_record['FireMask'].numpy().reshape(64, 64)
  prev_fire_mask = parsed_record['PrevFireMask'].numpy().reshape(64, 64)

# Convert the lists of data into numpy arrays
features_data = np.array(features_data)

X_transposed = features_data.transpose(1, 2, 0)
X_final = X_transposed.reshape(-1, 12)

# Convert the lists of data into numpy arrays
X_single = np.array(X_final)
y_single = np.array(labels_data).flatten()

print(f"Shape of features (X): {X_single.shape}")
print(f"Shape of labels (y): {y_single.shape}")

In [ ]:
single_record_prediction = lr_model.predict(X_single)

plt.figure(figsize=(10, 10))

plt.subplot(2,2,1)
plt.title(f"Prediction of record {1+num_records_to_skip} FireMask")
plt.imshow(single_record_prediction.reshape(64,64), cmap="Reds")
plt.colorbar()

plt.subplot(2,2,2)
plt.title(f"Record {1+num_records_to_skip} FireMask")
plt.imshow(fire_mask, cmap="Reds")
plt.colorbar()

plt.subplot(2,2,3)
plt.title(f"Record {1+num_records_to_skip} PrevFireMask")
plt.imshow(prev_fire_mask, cmap="Reds")
plt.colorbar()

plt.show()